# Fractional Ranking — Pandas `rank(method='average')`

Shows that the formula:

$$
r^s_j = 1 + \sum_{\substack{i=1 \\ i \neq j}}^{6} \left[ \mathbf{1}(\alpha_j^s > \alpha_i^s) + \tfrac{1}{2}\mathbf{1}(\alpha_j^s = \alpha_i^s) \right]
$$

reproduces `pd.Series.rank(method='average')` exactly, including ties.

In [1]:
import numpy as np
import pandas as pd

def fractional_rank(values):
    """Formula-based ranking matching pandas rank(method='average')."""
    n = len(values)
    ranks = np.zeros(n)
    for j in range(n):
        r = 1.0
        for i in range(n):
            if i == j:
                continue
            if values[j] > values[i]:
                r += 1.0
            elif values[j] == values[i]:
                r += 0.5
        ranks[j] = r
    return ranks

### No ties

In [2]:
alphas = np.array([3.1, 1.5, 4.2, 2.0, 5.8, 0.7])
regions = [f"Region {i+1}" for i in range(6)]

df = pd.DataFrame({"region": regions, "alpha": alphas})
df["pandas_rank"] = df["alpha"].rank(method="average")
df["formula_rank"] = fractional_rank(alphas)

assert np.allclose(df["pandas_rank"], df["formula_rank"]), "Mismatch!"
df

,region,alpha,pandas_rank,formula_rank
0,Region 1,3.1,4.0,4.0
1,Region 2,1.5,2.0,2.0
2,Region 3,4.2,5.0,5.0
3,Region 4,2.0,3.0,3.0
4,Region 5,5.8,6.0,6.0
5,Region 6,0.7,1.0,1.0


### Two-way tie

In [3]:
alphas_tie2 = np.array([3.0, 1.5, 3.0, 2.0, 5.8, 0.7])

df2 = pd.DataFrame({"region": regions, "alpha": alphas_tie2})
df2["pandas_rank"] = df2["alpha"].rank(method="average")
df2["formula_rank"] = fractional_rank(alphas_tie2)

assert np.allclose(df2["pandas_rank"], df2["formula_rank"]), "Mismatch!"
df2

,region,alpha,pandas_rank,formula_rank
0,Region 1,3.0,4.5,4.5
1,Region 2,1.5,2.0,2.0
2,Region 3,3.0,4.5,4.5
3,Region 4,2.0,3.0,3.0
4,Region 5,5.8,6.0,6.0
5,Region 6,0.7,1.0,1.0


### Three-way tie

In [4]:
alphas_tie3 = np.array([3.0, 3.0, 3.0, 2.0, 5.8, 0.7])

df3 = pd.DataFrame({"region": regions, "alpha": alphas_tie3})
df3["pandas_rank"] = df3["alpha"].rank(method="average")
df3["formula_rank"] = fractional_rank(alphas_tie3)

assert np.allclose(df3["pandas_rank"], df3["formula_rank"]), "Mismatch!"
print("Three-way tie for positions 3,4,5 → each gets rank", df3["pandas_rank"].iloc[0])
df3

Three-way tie for positions 3,4,5 → each gets rank 4.0


,region,alpha,pandas_rank,formula_rank
0,Region 1,3.0,4.0,4.0
1,Region 2,3.0,4.0,4.0
2,Region 3,3.0,4.0,4.0
3,Region 4,2.0,2.0,2.0
4,Region 5,5.8,6.0,6.0
5,Region 6,0.7,1.0,1.0


### Four-way tie

In [5]:
alphas_tie3 = np.array([3.0, 3.0, 3.0, 2.0, 3.0, 0.7])

df3 = pd.DataFrame({"region": regions, "alpha": alphas_tie3})
df3["pandas_rank"] = df3["alpha"].rank(method="average")
df3["formula_rank"] = fractional_rank(alphas_tie3)

assert np.allclose(df3["pandas_rank"], df3["formula_rank"]), "Mismatch!"
print("Three-way tie for positions 3,4,5 → each gets rank", df3["pandas_rank"].iloc[0])
df3

Three-way tie for positions 3,4,5 → each gets rank 4.5


,region,alpha,pandas_rank,formula_rank
0,Region 1,3.0,4.5,4.5
1,Region 2,3.0,4.5,4.5
2,Region 3,3.0,4.5,4.5
3,Region 4,2.0,2.0,2.0
4,Region 5,3.0,4.5,4.5
5,Region 6,0.7,1.0,1.0
